In [15]:
import torch
import torch.nn as nn
class SimpleCNN(nn.Module):
    def __init__(self,num_classes=4):
        super(SimpleCNN,self).__init__()   #(1,128,128)
        self.conv1 = nn.Conv2d(in_channels=1,out_channels=16,kernel_size=3,stride=2,padding=1)#(1,128/2,128/2)
        self.conv2 = nn.Conv2d(in_channels=16,out_channels=32,kernel_size=3,stride=2,padding=1) #(16,64/2.64/2)
        self.fc1=nn.Linear(32*32*32,128)  #(32,32,32)
        self.fc2=nn.Linear(128,num_classes)
        self.relu=nn.ReLU()
    def forward(self,x):
        x=self.relu(self.conv1(x))
        x=self.relu(self.conv2(x))
        x=x.view(x.size(0),-1)
        x=self.relu(self.fc1(x))
        x=self.fc2(x)
        return x
model=SimpleCNN()
print(model)
print(model(torch.randn(1,1,128,128)))
        

SimpleCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (fc1): Linear(in_features=32768, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=4, bias=True)
  (relu): ReLU()
)
tensor([[ 0.1201, -0.0116, -0.0403, -0.0517]], grad_fn=<AddmmBackward0>)


In [16]:
#Resnet 
import torchvision.transforms as transforms
import torchvision.models as models
class Resnet18(nn.Module):
    def __init__(self,num_classes=4):
        super(Resnet18,self).__init__()
        self.resnet = models.resnet18(pretrained=True)
        self.resnet.conv1 = nn.Conv2d(
            9, 64, kernel_size=7, stride=2, padding=3, bias=False
        )
        self.resnet.fc =nn.Linear(512,num_classes)
    def forward(self,x):
        x=self.resnet(x)
        return x
model=Resnet18()
criterion = nn.CrossEntropyLoss()
print(model(torch.randn(1,9,128,128)))

    

tensor([[ 0.2702, -0.0663,  0.1852,  0.2650]], grad_fn=<AddmmBackward0>)


In [17]:
#Mini Resnet from scratch

import torch.nn.functional as F
class BasicBlock(nn.Module):
    def __init__(self,in_channels,out_channels,stride=1):
        super(BasicBlock,self).__init__()
        self.conv1 = nn.Conv2d(in_channels,out_channels,3,stride=stride,padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels,out_channels,3,stride=1,padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut =  nn.Sequential()
        if stride !=1 or in_channels!=out_channels:
            self.shortcut =nn.Sequential(
                nn.Conv2d(in_channels,out_channels,1,stride =stride,padding=0),
                nn.BatchNorm2d(out_channels)
            )
    def forward(self,x):
        identity =x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(identity)
        return F.relu(out)

class MiniResnet(nn.Module):
    def __init__(self,num_classes=4):
        super(MiniResnet,self).__init__()
        self.conv1 = nn.Conv2d(1,64,3,stride=1,padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(64,64,2,stride=1)
        self.layer2 = self._make_layer(64,128,2,stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(128,num_classes)
    def _make_layer(self,in_channels,out_channels,blocks,stride):
        layers=[]
        layers.append(BasicBlock(in_channels,out_channels,stride))
        for _ in range(1,blocks):
            layers.append(BasicBlock(out_channels,out_channels))
        return nn.Sequential(*layers)
    def forward(self,x):
        x=F.relu(self.bn1(self.conv1(x)))
        x=self.layer1(x)
        x=self.layer2(x)
        x=self.avgpool(x)
        x=x.view(x.size(0),-1)
        x=self.fc(x)
        return x
model = MiniResnet()
print(model(torch.randn(1,1,128,128)))
# print(model)

tensor([[0.0214, 0.4039, 0.0489, 0.1985]], grad_fn=<AddmmBackward0>)


In [18]:
#Mini VIT
class PatchEmbedding(nn.Module):
    def __init__(self,img_size=128,patch_size=16,in_channels=1,embed_dim=64):
        super(PatchEmbedding,self).__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size//patch_size)**2
        self.proj = nn.Conv2d(in_channels,embed_dim,kernel_size=patch_size,stride=patch_size)
    def forward(self,x):
        x=self.proj(x)
        x=x.flatten(2)
        x=x.transpose(1,2)
        return x
        
class PositionalEncoding(nn.Module):
    def __init__(self,num_patches,embed_dim):
        super(PositionalEncoding,self).__init__()
        self.pos_embedding = nn.Parameter(torch.zeros(1,num_patches,embed_dim))
    def forward(self,x):
        return x + self.pos_embedding


class TransformerBlock(nn.Module):
    def __init__(self,embed_dim,num_heads,mlp_dim,dropout=0.1):
        super(TransformerBlock,self).__init__()
        self.attn = nn.MultiheadAttention(embed_dim,num_heads,dropout=dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim,mlp_dim),
            nn.ReLU(),
            nn.Linear(mlp_dim,embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)
    def forward(self,x):
        attn_output,_ =self.attn(x,x,x)
        x=self.norm1(x+attn_output)
        mlp_output =self.mlp(x)
        x= self.norm2(x+mlp_output)
        return x

class MiniVit(nn.Module):
    def __init__(self,img_size=128,patch_size=16,in_channels=1,embed_dim=64,num_heads=4,mlp_dim=128,num_classes=4,num_blocks=4):
        super(MiniVit,self).__init__()
        self.patch_embed = PatchEmbedding(img_size,patch_size,in_channels,embed_dim)
        self.pos_embed = PositionalEncoding(self.patch_embed.num_patches,embed_dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim,num_heads,mlp_dim) for _ in range(num_blocks)
        ])
        self.classifier = nn.Linear(embed_dim,num_classes)
    def forward(self,x):
        x=self.patch_embed(x)
        x=self.pos_embed(x)
        for block in self.blocks:
            x=block(x)
        x=x.mean(dim=1)
        x=self.classifier(x)
        return x

model =MiniVit()
print(model(torch.randn( 1,1, 128, 128)))
        

tensor([[ 0.1374, -0.1220,  0.3523, -0.3201]], grad_fn=<AddmmBackward0>)
